# 🔑 Solutions: Lesson 1: Data Cleaning I
**Try the problems yourself first!** Looking at a solution before struggling is like watching someone else do push-ups.
Every solution below is self-contained and has a short explanation of *why* it works.

In [1]:
import sys
sys.path.append("../utils")            # lets us import course_utils.py and agri_prep.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course_utils import *             # load_raw_farm, profile, setup_plots, ...

setup_plots()
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
np.random.seed(42)

---
### Problem 1.1 &nbsp; ⭐ &nbsp; Rescue the fertilizer column

Load the raw farm data. The column `fertilizer_kg_ha` contains values like `'85'`, `'85 kg'`, `'85kg/ha'`, `'unknown'` and `'-'`. (There were also `'N/A'` strings in the file, but `read_csv` already turned those into NaN when loading; a good reminder that `read_csv` silently treats some strings as missing.)

1. List the **distinct strings that contain no digits** at all.
2. Convert the column to a proper numeric column (`float`).
3. Report how many values are missing **before** and **after** conversion. Should they differ? Why?

In [2]:
raw = load_raw_farm()
s = raw["fertilizer_kg_ha"]

no_digits = s.dropna()[~s.dropna().str.contains(r"\d")]
print("Strings without digits:", no_digits.unique())

before = s.isna().sum()
num = pd.to_numeric(s.str.extract(r"(\d+\.?\d*)", expand=False))
after = num.isna().sum()
print(f"missing before: {before}   after: {after}   difference: {after - before} (= number of placeholder strings: {len(no_digits)})")
print(num.describe().round(1))

Strings without digits: <StringArray>
['-', 'unknown']
Length: 2, dtype: str
missing before: 260   after: 341   difference: 81 (= number of placeholder strings: 81)
count    2144.0
mean       76.0
std       145.3
min         0.0
25%        33.0
50%        70.0
75%       101.0
max      2955.0
Name: fertilizer_kg_ha, dtype: float64


**💬 Why this works:** Regular expressions let you rescue the number from messy text. The increase in missing values is *honest*: those cells never contained a number. Always check what `coerce`/extraction turned into NaN.

---
### Problem 1.2 &nbsp; ⭐⭐ &nbsp; Dates and impossible timelines

Convert `planting_date` and `harvest_date` to real datetimes. Then:

1. How many rows have a **harvest date earlier than the planting date**?
2. What is the minimum and maximum planting date?
3. Show 3 of the offending rows.

In [3]:
raw = load_raw_farm()
for c in ["planting_date", "harvest_date"]:
    raw[c] = pd.to_datetime(raw[c], format="mixed")

bad = raw["harvest_date"] < raw["planting_date"]
print("harvest before planting:", bad.sum())
print("planting range:", raw["planting_date"].min().date(), "->", raw["planting_date"].max().date())
raw.loc[bad, ["farm_id", "planting_date", "harvest_date"]].head(3)

harvest before planting: 15
planting range: 2019-03-01 -> 2023-11-27


,farm_id,planting_date,harvest_date
164,F0567,2020-10-17,2018-01-15
495,F0470,2019-10-01,2018-01-15
809,F0185,2020-11-26,2018-01-15


**💬 Why this works:** A cross-field rule (harvest must follow planting) catches errors that no single-column check can. These rows have a corrupted date: we'll treat them as invalid in Lesson 2.

---
### Problem 1.3 &nbsp; ⭐⭐ &nbsp; Hunt the hidden duplicates

1. How many rows are *exact* duplicates of an earlier row (`df.duplicated()`)?
2. Now convert the dates (see Problem 1.2) and count rows that repeat the **natural key** `farm_id + planting_date`.
3. Why is the second number larger? Show one pair of "near duplicates" side by side.

In [4]:
raw = load_raw_farm()
print("exact duplicates:", raw.duplicated().sum())

for c in ["planting_date", "harvest_date"]:
    raw[c] = pd.to_datetime(raw[c], format="mixed")
key = ["farm_id", "planting_date"]
print("key duplicates  :", raw.duplicated(subset=key).sum())

dups = raw[raw.duplicated(subset=key, keep=False)].sort_values(key)
# find a record that is NOT an exact copy of anything, then show everything sharing its key
first_near = dups[~dups.duplicated(keep=False)].iloc[0]
same_key = (dups["farm_id"] == first_near["farm_id"]) & (dups["planting_date"] == first_near["planting_date"])
dups[same_key].T

exact duplicates: 60
key duplicates  : 85


,1433,2309
farm_id,F0016,F0016
county,Trans-Nzoia,trans-nzoia
soil_type,Clay,Clay
farm_size_ha,1.93,1.93
experience,Expert,Expert
crop,Corn,CORN
season,Long rains,Long rains
planting_date,2022-04-21 00:00:00,2022-04-21 00:00:00
harvest_date,2022-10-03 00:00:00,2022-10-03 00:00:00
rainfall_mm,516.8,516.8


**💬 Why this works:** Exact matching misses records typed slightly differently ('MAIZE' vs 'Maize'). A *natural key* (the columns that identify one real-world event) finds them. This is why we convert types and standardise text *before* de-duplicating.

---
### Problem 1.4 &nbsp; ⭐⭐ &nbsp; Is the missingness random?

Convert fertilizer to numeric (as in Problem 1.1). Compare the **percentage of missing fertilizer values** between small farms (`farm_size_ha < 1`) and larger farms (`>= 1`).

Is this missing data MCAR, MAR or MNAR? What does it imply for how we should fill it?

In [5]:
raw = load_raw_farm()
raw["fertilizer_kg_ha"] = pd.to_numeric(raw["fertilizer_kg_ha"].str.extract(r"(\d+\.?\d*)", expand=False))

is_small = raw["farm_size_ha"] < 1
rate = raw.groupby(is_small)["fertilizer_kg_ha"].apply(lambda s: s.isna().mean() * 100).round(1)
rate.index = ["farm_size >= 1 ha", "farm_size < 1 ha"]
print("% of fertilizer values missing:")
print(rate.to_string())

% of fertilizer values missing:
farm_size >= 1 ha     4.8
farm_size < 1 ha     23.9


**💬 Why this works:** The rate is several times higher for small farms, so missingness depends on an observed variable: **MAR**. A global mean would ignore this; a group-wise fill or an `is_missing` flag lets the model use the pattern.

---
### Problem 1.5 &nbsp; ⭐⭐⭐ &nbsp; Which imputation strategy is best? Run an experiment!

We can *test* imputation by hiding values we actually know.

Using `rainfall_mm` (keep only rows where it is present and > 0):
1. Clean the county text with `s.str.lower().str.replace(r'[^a-z]', '', regex=True)`.
2. Randomly hide 15% of the values (use `np.random.default_rng(0)`).
3. Fill them three ways: **global mean**, **global median**, **county median** (learned from the *visible* values only).
4. Compute the RMSE of each strategy against the truth. Which wins, and why?

In [6]:
raw = load_raw_farm()
d = raw[raw["rainfall_mm"] > 0].copy()
d["county"] = d["county"].str.lower().str.replace(r"[^a-z]", "", regex=True)

rng = np.random.default_rng(0)
mask = rng.random(len(d)) < 0.15
truth = d.loc[mask, "rainfall_mm"]
visible = d["rainfall_mm"].where(~mask)

preds = {
    "global mean":   pd.Series(visible.mean(), index=d.index),
    "global median": pd.Series(visible.median(), index=d.index),
    "county median": visible.groupby(d["county"]).transform("median"),
}
for name, p in preds.items():
    rmse = np.sqrt(((p[mask] - truth) ** 2).mean())
    print(f"{name:14s} RMSE = {rmse:6.1f} mm")

global mean    RMSE =  178.9 mm
global median  RMSE =  178.7 mm
county median  RMSE =  115.8 mm


**💬 Why this works:** Rainfall differs a lot between counties (400 mm to 900 mm), so the county median is a far better guess than one number for everybody. **Imputation quality depends on how well the other columns explain the missing one.** And notice the method: we *measured* the options instead of guessing.